In [1]:
import asyncio
import json
import socket
import httpx
from langgraph_sdk import get_client
from langchain_core.output_parsers import StrOutputParser

LANGGRAPH_URL = "http://127.0.0.1:2025"
A2A_HOST = "127.0.0.1"
A2A_PORT = 2025
A2A_URL = f"http://{A2A_HOST}:{A2A_PORT}"
A2A_RPC_PATH = "/a2a/"

# No enviamos automáticamente claves de LangSmith al servidor local.
lg = get_client(url=LANGGRAPH_URL, api_key=None, timeout=120)

def mostrar(valor):
    print(json.dumps(valor, indent=2, ensure_ascii=False, default=str))

# Encontrar el Id de tu agente

In [2]:
# El formato se fija para que el listado no dependa del valor por defecto del SDK.
asistentes = await lg.assistants.search(limit=100, response_format="array")
for asistente in asistentes:
    print({
        "name": asistente.get("name"),
        "graph_id": asistente.get("graph_id"),
        "assistant_id": asistente.get("assistant_id"),
    })
if not asistentes:
    raise RuntimeError("El servidor no devolvió asistentes. Revisa langgraph.json.")

{'name': 'agente_czz', 'graph_id': 'agente_czz', 'assistant_id': '3d58b072-566f-59e7-90f2-8fb84b2a11fe'}


In [3]:
ASSISTANT_ID = '3d58b072-566f-59e7-90f2-8fb84b2a11fe'

# Comunicacion directo con el agente

In [4]:
OUTPUT_KEY = None  # Ejemplo: "respuesta" si tu salida no utiliza messages.

def crear_entrada(texto):
    return {"messages": [{"role": "human", "content": texto}]}


# Extraer respuesta del json gigantesco del grafo
def extraer_texto(estado):
    if not isinstance(estado, dict):
        raise ValueError("La salida no es un diccionario; adapta extraer_texto a tu grafo.")
    if estado.get("__interrupt__"):
        raise RuntimeError("Tu grafo quedó pausado: resuelve el interrupt antes de responder.")
    if OUTPUT_KEY:
        contenido = estado[OUTPUT_KEY]
    else:
        # No tomamos el mensaje humano como si fuera una respuesta del agente.
        mensajes = estado.get("messages", [])
        respuesta = next(
            (m for m in reversed(mensajes)
             if m.get("type", m.get("role")) in ("ai", "assistant")),
            None,
        )
        if respuesta is None:
            raise ValueError("No hay respuesta ai/assistant. Configura OUTPUT_KEY.")
        contenido = respuesta.get("content", "")
    if isinstance(contenido, list):
        contenido = "\n".join(
            bloque if isinstance(bloque, str) else bloque.get("text", "")
            for bloque in contenido
            if isinstance(bloque, (str, dict))
        )
    if not isinstance(contenido, str) or not contenido.strip():
        raise ValueError("La salida no tiene texto. Adapta extraer_texto a tu agente.")
    return contenido


async def consultar_langgraph(texto, thread_id=None):
    estado = await lg.runs.wait(
        thread_id,
        ASSISTANT_ID,
        input=crear_entrada(texto),
    )
    return extraer_texto(estado)

In [5]:
# Verificación 1 · personalidad: debe usar mi nombre, mi especialidad y mi tono.
PREGUNTA = "Presentate y cuentame como me puedes ayudar"
estado = await lg.runs.wait(None, ASSISTANT_ID, input=crear_entrada(PREGUNTA))
print("\nRespuesta:")
print(extraer_texto(estado))


Respuesta:
¡Hola! Soy el agente de Carlos Zapata. Estoy aquí para ayudarte a ponerle rigor y datos concretos al análisis de tus inversiones. 

Como especialista en análisis de acciones, mi foco es facilitarte el trabajo antes de que decidas mover un peso. ¿En qué te puedo ayudar? Aquí tienes lo que mejor sé hacer:

1. **Consultar datos de mercado en tiempo real:** Precios, variaciones del día y moneda local de las acciones que te interesen (solo necesito que me des el *ticker* y el país del mercado).
2. **Revisar tus apuntes:** Tengo acceso a tus notas sobre conceptos, criterios y casos para recordar qué metodologías sueles usar.
3. **Analizar fundamentos:** Te puedo ayudar a repasar métricas clave como el PER, evaluar el riesgo o pensar en cómo diversificar tu cartera con ejemplos simples.
4. **Armar un plan paso a paso:** Diseñar la ruta de revisión que deberías seguir antes de evaluar cualquier alternativa de inversión.

Eso sí, ten claro que **yo no ejecuto compras, ventas ni tran

In [6]:
# Verificación 2 · conocimientos y skills: debe consultar mis apuntes y aplicar una habilidad.
PREGUNTA_APUNTES = ("Usa el caso de mis apuntes y aplica una de tus habilidades para proponer "
                    "un plan de analisis antes de invertir en una accion chilena.")
print(">", PREGUNTA_APUNTES, "\n")
print(await consultar_langgraph(PREGUNTA_APUNTES))

> Usa el caso de mis apuntes y aplica una de tus habilidades para proponer un plan de analisis antes de invertir en una accion chilena. 

Basándome en el caso de nuestros apuntes —donde una empresa con buenos números tenía una observación tributaria pendiente—, te propongo un plan de análisis paso a paso antes de poner un peso en una acción chilena:

1. **Análisis fundamental (mis apuntes y datos duros):** Revisa el negocio, su endeudamiento, flujos y si el PER (Precio/Utilidad) está razonable frente a su historia y la industria.
2. **Chequeo normativo y tributario (lección del caso):** No te quedes solo en los números. Revisa memorias, estados financieros y notas explicativas para detectar juicios, multas o contingencias fiscales pendientes.
3. **Consulta con expertos (alcance y límites):** Recuerda que los temas estrictamente tributarios o legales escapan de nuestro análisis financiero. Si detectas alguna alerta fiscal, **derívala siempre a un especialista** (abogado o contador audit

# Memoria

In [7]:
ASSISTANT_ID = '3d58b072-566f-59e7-90f2-8fb84b2a11fe'

# Verificación 3 · memoria: mismo thread_id en los dos turnos.
thread = await lg.threads.create()
thread_id = thread["thread_id"]
print(await consultar_langgraph("Mi objetivo para este proyecto es invertir 5 millones de pesos en acciones chilenas.", thread_id=thread_id))
print()
print(await consultar_langgraph("¿Qué objetivo te acabo de contar?", thread_id=thread_id))

No tengo registros específicos sobre "acciones chilenas" en tus apuntes actuales, pero basándome en mi especialidad y en los criterios que debemos seguir, aquí no hay espacio para la improvisación. 

Antes de decidir dónde colocar esos 5 millones, propongo seguir este plan paso a paso:

1. **Definir el horizonte de tiempo y perfil de riesgo:** ¿Es plata que vas a necesitar en 1 año o en 10? ¿Aguantas ver tu portafolio caer un 15% en un mes sin entrar en pánico?
2. **Diversificación:** Nunca pongas los 5 millones en una sola acción chilena. Hay que repartir el riesgo entre distintos sectores (por ejemplo, banca, retail, commodities o servicios básicos).
3. **Análisis fundamental:** Revisar los números duros de las empresas que te llamen la atención: valorización (como el PER), nivel de endeudamiento y generación de caja.

¿Tienes ya en la mira alguna empresa o sector en la bolsa chilena (por ejemplo, SQM, CMPC, Banco de Chile, etc.) para que empecemos a revisar sus datos de mercado? 

*

# Como se ve esto en langraph

In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class EstadoCoordinador(TypedDict):
    pregunta: str
    respuesta: str

consulta_remota = consultar_langgraph

async def delegar(estado: EstadoCoordinador):
    respuesta = await consulta_remota(estado["pregunta"])
    return {"respuesta": respuesta}

builder = StateGraph(EstadoCoordinador)
builder.add_node("delegar", delegar)

builder.add_edge(START, "delegar")
builder.add_edge("delegar", END)

coordinador = builder.compile()

resultado = await coordinador.ainvoke({"pregunta": "Hola que puedes hacer?"})
print(resultado["respuesta"])

¡Hola! Soy el asistente de Carlos Zapata. Te ayudo con el análisis fundamental de acciones y a revisar datos antes de tomar decisiones de inversión.

Esto es lo que puedo hacer por ti:
- **Consultar precios y datos de mercado** de acciones (para eso necesito el *ticker* y el país del mercado donde cotiza).
- **Revisar tus apuntes** para buscar conceptos, criterios o casos que tengas guardados.
- **Explicar conceptos de valorización**, como el PER, el riesgo o la diversificación, con ejemplos simples y directos.
- **Proponer un plan de análisis paso a paso** antes de invertir.

*(Ojo: No hago ejecuciones de compra o venta, ni doy asesoría tributaria o legal; eso le corresponde a especialistas).*

¿En qué te puedo ayudar hoy?


In [9]:
print(coordinador.get_graph().draw_mermaid())            # texto, siempre funciona


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	delegar(delegar)
	__end__([<p>__end__</p>]):::last
	__start__ --> delegar;
	delegar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



# Vamos a Agent2Agent.

In [10]:
# pip install a2a-sdk
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Role, SendMessageRequest
from a2a.helpers import get_message_text
import httpx 
from google.protobuf.json_format import MessageToDict

http_a2a = httpx.AsyncClient(timeout=120, trust_env=False)
resolver = A2ACardResolver(http_a2a, A2A_URL)
card_remota = await resolver.get_agent_card(
    http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
)
print("Ficha descargada:")
mostrar(MessageToDict(card_remota))

# El servidor anuncia protocolVersion 1.0 pero responde con el JSON de A2A 0.3:
# se fija 0.3.0 para que el cliente use el transporte compatible.
for interfaz in card_remota.supported_interfaces:
    interfaz.protocol_version = "0.3.0"

cliente_a2a = await create_client(
    card_remota,
    client_config=ClientConfig(
        httpx_client=http_a2a,
        streaming=False,
        supported_protocol_bindings=["JSONRPC"],
    ),
)
print("Cliente preparado con la Agent Card.")

# el cliente se usa en la celda siguiente y se cierra al final de la etapa


Ficha descargada:
{
  "name": "agente_czz",
  "description": "agente_czz assistant",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2025/a2a/3d58b072-566f-59e7-90f2-8fb84b2a11fe",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "0.16.5",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "Choose task-only or full-context response history",
        "params": {
          "default": "context",
          "methods": [
            "SendMessage",
            "GetTask",
            "ListTasks"
          ],
          "values": [
            "context",
            "task"
          ],
          "header": "LangGraph-A2A-History-Scope"
        }
      }
    ]
  },
  "defaultInputModes": [
    "application/json",
    "text/plain"
  ],
  "defaultOutputModes": [
    "application/json",
    "text/pl

In [11]:
# Verificación 4 · un envio real por A2A (JSONRPC) al mismo agente.
from a2a.helpers import new_text_message, get_stream_response_text, get_message_text
from a2a.types import Role, SendMessageRequest


async def enviar_a2a(cliente, texto, contexto=None):
    peticion = SendMessageRequest(message=new_text_message(
        texto, role=Role.ROLE_USER, context_id=contexto))
    partes = []
    async for evento in cliente.send_message(peticion):
        fragmento = get_stream_response_text(evento)
        if evento.HasField("task"):
            contexto = evento.task.context_id or contexto
            if not fragmento and evento.task.status.HasField("message"):
                fragmento = get_message_text(evento.task.status.message)
        elif evento.HasField("message"):
            contexto = evento.message.context_id or contexto
        if fragmento:
            partes.append(fragmento)
    if not partes:
        raise RuntimeError("El agente no devolvio texto. Revisa su terminal.")
    return "\n".join(partes), contexto


respuesta_a2a, contexto_a2a = await enviar_a2a(
    cliente_a2a, "Cuanto vale AAPL en el mercado US?")
print(respuesta_a2a)

await cliente_a2a.close()
await http_a2a.aclose()

El precio actual de **Apple Inc. (AAPL)** en el mercado de EE.UU. (NASDAQ) es de **$342.10 USD** (unos **$331.250 CLP**), con una variación del día de **+1.24%**. 

Recuerda que esto es solo información de mercado; si estás evaluando entrar, lo prudente es revisar antes los fundamentos y ver cómo encaja en tu estrategia. ¿Quieres que revise algo más sobre esta acción o prefieres que veamos sus métricas?


# Etapa 3 · Dos agentes del grupo conversando por A2A

El reto del taller es **evaluar si conviene invertir**. Lo responden dos agentes de dos
integrantes, cada uno en su propio servidor: `agente_czz` (acciones, Carlos Zapata) en el
puerto 2025 y `agente_bryhan` (procesos e IA, Bryhan Cortes) en el 2026.

Un **supervisor** descubre las dos Agent Cards, elige quién responde cada turno y envía el
mensaje por A2A. Cada agente conserva su propio `contextId`, así que recuerda lo que ya
conversó en los turnos anteriores.

> **Compañeros en otra red.** El proyecto trae `tunel_cloudflare.ps1` (y `tunel_cloudflare.py`)
> para publicar el puerto 2025 con un túnel temporal. Exporta la URL pública antes de ejecutar
> la etapa 3 (`$env:A2A_CZZ_URL` / `$env:A2A_BRYHAN_URL`): la celda de configuración la toma y la
> de descubrimiento reescribe en la tarjeta la URL del túnel.


In [12]:
# ── ETAPA 3 · 0. CONFIGURACIÓN ────────────────────────────────────────────
# Dos agentes de dos integrantes del grupo, cada uno en su propio servidor A2A.
# Si un agente está publicado con túnel de Cloudflare, basta exportar su URL:
#   $env:A2A_CZZ_URL = "https://<algo>.trycloudflare.com"
#   $env:A2A_BRYHAN_URL = "https://<algo>.trycloudflare.com"
# Sin esas variables el notebook usa 127.0.0.1 (misma máquina).
import os
from pathlib import Path
from typing import TypedDict
from uuid import uuid4

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.helpers import get_message_text, get_stream_response_text, new_text_message
from a2a.types import Role, SendMessageRequest


def buscar_env(inicio: Path) -> Path | None:
    """Busca .env en la carpeta actual, en sus superiores o en mi-agente-cz/."""
    for carpeta in (inicio, *inicio.parents):
        for ruta in (carpeta / ".env", carpeta / "mi-agente-cz" / ".env"):
            if ruta.is_file():
                return ruta
    return None


ENV = buscar_env(Path.cwd().resolve())
if ENV:
    load_dotenv(ENV, override=False)
    print("Entorno cargado desde:", ENV)
else:
    print("Aviso: no encontré un .env; exporta GOOGLE_API_KEY en el kernel.")

# El supervisor usa Gemini: sin clave no puede decidir con quién conversar.
if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError(
        "Falta GOOGLE_API_KEY. Agrégala a mi-agente-cz/.env o expórtala en el kernel "
        "antes de ejecutar la etapa 3.")

SERVIDORES = {
    os.environ.get("A2A_CZZ_URL") or "http://127.0.0.1:2025": "Carlos Zapata",   # agente_czz
    os.environ.get("A2A_BRYHAN_URL") or "http://127.0.0.1:2026": "Bryhan Cortes",  # agente_bryhan
}
print("Servidores a descubrir:", list(SERVIDORES))


Entorno cargado desde: C:\Users\Carlos Zapata Z\Downloads\UAI - Certificado Agentic AI\Taller UAI IA\grupo-1-taller-a2a\mi-agente-cz\.env
Servidores a descubrir: ['http://127.0.0.1:2025', 'http://127.0.0.1:2026']


In [13]:
# ── 1. DESCUBRIR LOS DOS AGENTES Y PREPARAR UN CLIENTE A2A POR AGENTE ────
http_comun = httpx.AsyncClient(timeout=180, trust_env=False)
AGENTES = {}      # nombre del grafo -> cliente A2A, contexto propio y tarjeta

# La tarjeta es la carta de presentacion del agente: aqui se escribe lo que sabe hacer,
# porque de eso depende que el supervisor elija bien. Es idempotente: se puede repetir.
FICHAS = {
    "agente_czz": (
        "Agente de Carlos Zapata. Analiza acciones e inversiones: precios de acciones de "
        "EE.UU. y Chile, criterios y apuntes de inversion, casos documentados. Responde "
        "preguntas sobre empresas, tickers y decisiones de inversion en bolsa."),
    "agente_bryhan": (
        "Agente de Bryhan Cortes. AI Engineering aplicado a procesos empresariales y "
        "financieros: analiza procesos, disena soluciones de datos e IA, compara "
        "alternativas tecnicas y arma planes de trabajo."),
}

for base, autor in SERVIDORES.items():
    asistentes = httpx.post(f"{base}/assistants/search", json={"limit": 10},
                            timeout=60, trust_env=False).json()
    for asistente in asistentes:
        if asistente["graph_id"] in FICHAS:
            httpx.patch(f"{base}/assistants/{asistente['assistant_id']}",
                        json={"name": asistente["graph_id"],
                              "description": FICHAS[asistente["graph_id"]]},
                        timeout=60, trust_env=False).raise_for_status()
        tarjeta = await A2ACardResolver(http_comun, base).get_agent_card(
            http_kwargs={"params": {"assistant_id": asistente["assistant_id"]}})
        # El servidor anuncia protocolVersion 1.0 pero responde con el JSON de A2A 0.3:
        # se fija 0.3.0 para que el cliente use el transporte compatible.
        # Ademas, la tarjeta anuncia la URL local (127.0.0.1): si el agente esta detras de un
        # tunel de Cloudflare, el cliente tiene que hablarle por la URL publica.
        url_a2a = None
        for interfaz in tarjeta.supported_interfaces:
            interfaz.protocol_version = "0.3.0"
            if interfaz.protocol_binding == "JSONRPC":
                interfaz.url = f"{base}/a2a/{asistente['assistant_id']}"
                url_a2a = interfaz.url
        cliente = await create_client(
            tarjeta,
            client_config=ClientConfig(httpx_client=http_comun, streaming=False,
                                       supported_protocol_bindings=["JSONRPC"]),
        )
        # Un contextId por agente: es su thread_id, asi recuerda entre turnos.
        AGENTES[tarjeta.name] = {
            "cliente": cliente,
            "contexto": str(uuid4()),
            "autor": autor,
            "descripcion": tarjeta.description,
            "url": url_a2a or tarjeta.supported_interfaces[0].url,
        }

print("Agentes A2A disponibles:", list(AGENTES))
for nombre, datos in AGENTES.items():
    print(f"  {nombre:<15} {datos['autor']:<15} {datos['url']}")


Agentes A2A disponibles: ['agente_czz', 'agente_bryhan']
  agente_czz      Carlos Zapata   http://127.0.0.1:2025/a2a/3d58b072-566f-59e7-90f2-8fb84b2a11fe
  agente_bryhan   Bryhan Cortes   http://127.0.0.1:2026/a2a/a4795f3b-403a-5782-ae56-443223f44e86


In [14]:
# ── 2. SUPERVISOR: ELIGE CON QUIÉN CONVERSAR ─────────────────────────────
class Decision(BaseModel):
    agente: str = Field(description="Nombre exacto de uno de los agentes del catálogo")
    motivo: str = Field(description="Por qué ese agente puede responder mejor la consulta")


modelo_router = ChatGoogleGenerativeAI(model=os.environ.get("MODELO", "gemini-3.5-flash-lite"),
                                       temperature=0)
enrutador = modelo_router.with_structured_output(Decision)

PLANTILLA_ROUTER = """Eres el supervisor de un grupo de agentes que conversan por A2A.
Elige UN agente del catálogo para responder la consulta del usuario.

Catálogo:
{catalogo}

Consulta del usuario:
<consulta>{consulta}</consulta>

Devuelve el nombre exacto del agente y un motivo breve."""


def catalogo() -> str:
    """Texto con los agentes disponibles y lo que sabe hacer cada uno."""
    return "\n".join(
        f"- {nombre}: {datos['descripcion']} (representa a {datos['autor']})"
        for nombre, datos in AGENTES.items())


print(catalogo())

- agente_czz: Agente de Carlos Zapata. Analiza acciones e inversiones: precios de acciones de EE.UU. y Chile, criterios y apuntes de inversion, casos documentados. Responde preguntas sobre empresas, tickers y decisiones de inversion en bolsa. (representa a Carlos Zapata)
- agente_bryhan: Agente de Bryhan Cortes. AI Engineering aplicado a procesos empresariales y financieros: analiza procesos, disena soluciones de datos e IA, compara alternativas tecnicas y arma planes de trabajo. (representa a Bryhan Cortes)


In [15]:
# ── 3. NODOS: EL SUPERVISOR Y LA LLAMADA A2A AL AGENTE ELEGIDO ──────────
class EstadoReunion(TypedDict):
    consulta: str
    agente: str
    motivo: str
    respuesta_final: str


async def nodo_supervisor(estado: EstadoReunion) -> dict:
    """Decide qué agente responde este turno."""
    decision = await enrutador.ainvoke(
        PLANTILLA_ROUTER.format(catalogo=catalogo(), consulta=estado["consulta"]))
    agente = decision.agente if decision.agente in AGENTES else next(iter(AGENTES))
    print(f"[supervisor] -> {agente} · {decision.motivo}")
    return {"agente": agente, "motivo": decision.motivo}


async def nodo_agente_remoto(estado: EstadoReunion) -> dict:
    """Envía la consulta por A2A al agente elegido y devuelve su respuesta."""
    datos = AGENTES[estado["agente"]]
    peticion = SendMessageRequest(
        message=new_text_message(estado["consulta"], context_id=datos["contexto"],
                                 role=Role.ROLE_USER))
    partes = []
    async for evento in datos["cliente"].send_message(peticion):
        texto = get_stream_response_text(evento)
        if not texto and evento.HasField("task") and evento.task.status.HasField("message"):
            texto = get_message_text(evento.task.status.message)
        if texto:
            partes.append(texto)
    respuesta = "\n".join(partes) or "(el agente no devolvió texto)"
    print(f"[{estado['agente']}] respondió {len(respuesta)} caracteres")
    return {"respuesta_final": respuesta}


print("Nodos listos: supervisor y agente_remoto.")

Nodos listos: supervisor y agente_remoto.


In [16]:
# ── 4. GRAFO DEL SUPERVISOR ─────────────────────────────────────────────
def construir_reunion():
    """supervisor elige el agente y el nodo remoto hace la llamada A2A."""
    flujo = StateGraph(EstadoReunion)
    flujo.add_node("supervisor", nodo_supervisor)
    flujo.add_node("agente_remoto", nodo_agente_remoto)
    flujo.add_edge(START, "supervisor")
    flujo.add_edge("supervisor", "agente_remoto")
    flujo.add_edge("agente_remoto", END)
    return flujo.compile()


reunion = construir_reunion()
print(reunion.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	agente_remoto(agente_remoto)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	supervisor --> agente_remoto;
	agente_remoto --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [17]:
# ── 5. CONVERSACIÓN DE TRES TURNOS SOBRE EL RETO DEL TALLER ─────────────
# Reto: "evaluar si conviene invertir". Los turnos 1 y 3 los responde el agente de
# acciones (Carlos); el turno 2, el de procesos e IA (Bryhan). El supervisor decide solo.
RETO = "Evaluar si conviene invertir en una empresa"

TURNOS = [
    "Reto del taller: evaluar si conviene invertir en Falabella. ¿Qué datos y qué "
    "criterios debería revisar antes de decidir?",
    "Soy estudiante y quiero automatizar ese análisis con IA: ¿cómo diseñaría el proceso "
    "y qué debería evitar?",
    "Con todo lo anterior, dame el criterio final: ¿cuándo conviene invertir y cuándo no?",
]

historial = []
for numero, consulta in enumerate(TURNOS, 1):
    print("=" * 78)
    print(f"TURNO {numero} · usuario: {consulta}")
    salida = await reunion.ainvoke(
        {"consulta": consulta, "agente": "", "motivo": "", "respuesta_final": ""})
    print(f"\n[respuesta de {salida['agente']}]\n{salida['respuesta_final']}\n")
    historial.append(salida)

print("=" * 78)
print("Conversación cerrada. Un contextId por agente:",
      {nombre: datos["contexto"][:8] + "…" for nombre, datos in AGENTES.items()})
for datos in AGENTES.values():
    await datos["cliente"].close()
await http_comun.aclose()

TURNO 1 · usuario: Reto del taller: evaluar si conviene invertir en Falabella. ¿Qué datos y qué criterios debería revisar antes de decidir?


C:\Users\Carlos Zapata Z\Downloads\UAI - Certificado Agentic AI\Taller UAI IA\grupo-1-taller-a2a\venv-notebook\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[supervisor] -> agente_czz · La consulta trata sobre evaluar una inversión en una empresa específica (Falabella), lo cual corresponde al análisis de acciones, criterios de inversión y decisiones en bolsa que maneja el agente de Carlos Zapata.


[agente_czz] respondió 1736 caracteres

[respuesta de agente_czz]
Como mis apuntes no tienen un registro específico para Falabella (ni de tus notas previas), te propongo el **plan de análisis paso a paso** que usamos en el Certificado de IA Agéntica antes de poner un peso en cualquier acción:

### 1. Datos de mercado actuales
Primero necesitamos el pulso del mercado. Para consultar la acción de Falabella en su mercado (Chile), dime el ticker exacto (suele ser FALABELLA en la Bolsa de Santiago, pero confírmame si lo tienes claro). Con eso consultamos:
* Precio actual y variación del día.
* Capitalización bursátil.

### 2. Criterios de análisis fundamental (Mis sugerencias y rigor técnico)
Antes de decidir, revisa estos tres pilares:
* **Valorización (PER y P/B):** ¿Estamos pagando un precio razonable por las utilidades y los activos de la empresa, o está cara en comparación con su historia y sus competidores (como Cencosud o Ripley)?
* **Salud Financiera (Deuda):** Falabella es un negoc

C:\Users\Carlos Zapata Z\Downloads\UAI - Certificado Agentic AI\Taller UAI IA\grupo-1-taller-a2a\venv-notebook\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[supervisor] -> agente_bryhan · Es experto en AI Engineering aplicado a procesos, diseño de soluciones de datos e IA, y análisis técnico, lo cual se alinea directamente con la necesidad del usuario de automatizar un proceso y saber qué evitar.


[agente_bryhan] respondió 3427 caracteres

[respuesta de agente_bryhan]
**Conclusión:** Para automatizar un análisis con IA de forma rigurosa, se debe estructurar el proceso desde el diagnóstico del problema hasta la validación de resultados, evitando el error común de aplicar IA generativa a problemas deterministas donde bastaría con reglas o automatización tradicional.

Como agente profesional de Bryhan, presento a continuación una guía de diseño y buenas prácticas alineada con su enfoque en automatización y analítica aplicada.

---

### 1. Cómo diseñar el proceso de automatización (Propuesta estructurada)

Basado en la metodología de diseño de soluciones de datos e IA, el proceso debe seguir cinco fases:

1. **Diagnóstico y Análisis del Proceso Actual:**
   * **Objetivo:** Definir con precisión qué decisión o análisis se quiere automatizar.
   * **Entradas y Salidas:** Identificar el formato de los datos de origen (texto, tablas, PDFs, bases de datos) y el entregable esperado.
   * 

C:\Users\Carlos Zapata Z\Downloads\UAI - Certificado Agentic AI\Taller UAI IA\grupo-1-taller-a2a\venv-notebook\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[supervisor] -> agente_czz · La consulta solicita criterios de inversión sobre cuándo conviene invertir y cuándo no, lo cual corresponde directamente al área de especialización de Carlos Zapata en acciones, inversiones y decisiones bursátiles.


[agente_czz] respondió 1610 caracteres

[respuesta de agente_czz]
Aquí vamos directo al grano. Basado en el análisis fundamental y la gestión de riesgo, el criterio final se reduce a esto:

### **¿Cuándo SÍ conviene invertir?**
1. **Valorización atractiva:** Cuando el precio de la acción está castigado (un PER o un Precio/Libro bajo su promedio histórico), pero **sin** que eso refleje un daño estructural en el negocio. Es decir, compras barato un activo bueno.
2. **Recuperación operativa comprobada:** Cuando los datos muestran que los márgenes mejoran, la deuda neta baja y la estrategia digital (e-commerce) por fin es rentable, no solo una promesa.
3. **Margen de seguridad:** Cuando el precio actual está muy por debajo del valor intrínseco estimado, protegiéndote si el mercado cae.

### **¿Cuándo NO conviene invertir?**
1. **Endeudamiento fuera de control:** Si el ratio de deuda sobre EBITDA sube demasiado en un escenario de tasas de interés altas, el costo financiero se come las utili